In [ ]:
import os
import h5py
import numpy as np
import pandas as pd

from sklearn.decomposition import PCA
import openTSNE
import glasbey

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

pio.renderers.default = "browser"

# -------------------------------------------------------------------
# 1. Load similarity matrix + cluster indices from .mat
# -------------------------------------------------------------------
mat_filename = r"X:\3darena_behavior\wildtype_062425\011725_9\011725_9_3d_arenah_070225\cluster_output\Combined_Results\Results\test1\session_1_out.mat"

if os.path.exists(mat_filename):
    with h5py.File(mat_filename, 'r') as file:
        clusters = file['Clusters']
        similarity_matrix = clusters['sim'][()]      # shape (N, D)
        cluster_labels = clusters['idx'][()]         # could be shape (1, N) or (N, 1), etc.
else:
    raise FileNotFoundError("Could not find session_1_out.mat in dedicated subdirectory")

print("similarity_matrix:", similarity_matrix.shape)
print("cluster_labels raw:", cluster_labels.shape)

# Make cluster_labels a 1D array of length N
cluster_labels = cluster_labels.T        # if shape was (1, N) or (N, 1)
cluster_labels = cluster_labels.flatten()
print("cluster_labels flattened:", cluster_labels.shape)





similarity_matrix: (47715, 47715)
cluster_labels raw: (1, 47715)
cluster_labels flattened: (47715,)
week_numbers shape: (47715,)
stages shape: (47715,)
similarity_matrix_reduced: (47715, 50)
embedding: (47715, 2)


In [ ]:

# df_combined must have columns: "Cluster" and "Week_Number"
# If there are multiple rows per cluster in combined_matrix, you may need
# to decide how to handle that (e.g., unique mapping).
# Here we assume each Cluster has exactly one Week_Number.
df_combined_unique = df_combined.drop_duplicates(subset=["Cluster"])

# Merge to attach Week_Number to each sample row based on Cluster index
df_merged = df_samples.merge(
    df_combined_unique[["Cluster", "Week_Number"]],
    on="Cluster",
    how="left"
)

if df_merged["Week_Number"].isna().any():
    raise ValueError("Some cluster labels could not be matched to Week_Number in combined_matrix.")

week_numbers = df_merged["Week_Number"].values  # shape (N,)
print("week_numbers shape:", week_numbers.shape)

# Optionally: map Week_Number (8,10,12,14,23) to stage labels 1..5
week_to_stage = {
    8: 1,
    10: 2,
    12: 3,
    14: 4,
    23: 5
}
stages = np.array([week_to_stage[w] for w in week_numbers])
print("stages shape:", stages.shape)

# -------------------------------------------------------------------
# 3. PCA on similarity_matrix
# -------------------------------------------------------------------
pca = PCA(n_components=50)
similarity_matrix_reduced = pca.fit_transform(similarity_matrix)
print("similarity_matrix_reduced:", similarity_matrix_reduced.shape)

# -------------------------------------------------------------------
# 4. t-SNE (example: PCA initialization + cosine)
# -------------------------------------------------------------------
embedding = openTSNE.TSNE(
    perplexity=30,
    initialization="pca",
    metric="cosine",
    n_jobs=8,
    random_state=3,
).fit(similarity_matrix_reduced)   # shape (N, 2)

print("embedding:", embedding.shape)

# -------------------------------------------------------------------
# 5. Color by stage (1..5) OR by Week_Number
# -------------------------------------------------------------------
# Choose what you want to use in the plot:
#   labels_for_color = stages      # 1..5
#   or:
#   labels_for_color = week_numbers   # 8,10,12,14,23
labels_for_color = stages

labels_str = labels_for_color.astype(str)
unique_labels = sorted(set(labels_str))

palette = glasbey.create_palette(palette_size=len(unique_labels))
label_to_color = {lab: palette[i] for i, lab in enumerate(unique_labels)}
colors = [label_to_color[lab] for lab in labels_str]

# -------------------------------------------------------------------
# 6. Plot with Plotly
# -------------------------------------------------------------------
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=embedding[:, 0],
        y=embedding[:, 1],
        mode="markers",
        marker=dict(size=5, color=colors),
        text=labels_str,  # hover text
        showlegend=False
    )
)

fig.update_layout(
    title="t-SNE of similarity matrix colored by stage",
    xaxis_title="t-SNE 1",
    yaxis_title="t-SNE 2"
)

fig.show()

In [8]:
import os
import h5py
import numpy as np
import pandas as pd

from sklearn.decomposition import PCA
import openTSNE

# -----------------------------
# 1. Read similarity + cluster_labels from .mat
# -----------------------------
mat_filename = r"Y:\Members\Mia-Sanjana-Hadent\Processed Data\wildtype_062425_011725_9\Altogether Clustering\session_1_out.mat"

if os.path.exists(mat_filename):
    with h5py.File(mat_filename, "r") as f:
        clusters = f["Clusters"]
        similarity_matrix = clusters["sim"][()]    # (N, D)
        cluster_labels = clusters["idx"][()]       # maybe (1, N) or (N, 1)
else:
    raise FileNotFoundError("Could not find session_1_out.mat")

cluster_labels = cluster_labels.T
cluster_labels = cluster_labels.flatten()
print("similarity_matrix:", similarity_matrix.shape)
print("cluster_labels:", cluster_labels.shape)

n_samples = similarity_matrix.shape[0]
if cluster_labels.shape[0] != n_samples:
    raise ValueError("cluster_labels length != similarity_matrix rows")




similarity_matrix: (119227, 119227)
cluster_labels: (119227,)


In [ ]:
# -----------------------------
# 2. Read combined_matrix with one row per sample
#    Example columns: 'cluster_idx', 'Week_Number'
# -----------------------------
combined_csv = r"Y:\Members\Mia-Sanjana-Hadent\Processed Data\wildtype_062425_011725_9\Altogether Clustering\Cluster_detail_results.csv"
combined = pd.read_csv(combined_csv)
print("combined_matrix rows:", len(combined))

if len(combined) != n_samples:
    raise ValueError(
        f"combined_matrix rows {len(combined)} != similarity_matrix rows {n_samples}"
    )

# If combined_matrix is already in the same order as cluster_labels,
# you can just assert they match:
if "cluster_idx" in combined.columns:
    if not np.array_equal(combined["cluster_idx"].values, cluster_labels):
        raise ValueError("cluster_idx column does not match cluster_labels order")

# Now you can get Week_Number directly
week_numbers = combined["Folder_Name"].values
print("week_numbers:", week_numbers.shape)

# Optional mapping Week_Number -> stage 1..5
week_to_stage = {"5_restricted_arena": 1, "4_open_arena": 2, "2_3D_arenaL": 3, "3_3D_arenaM": 4, "1_3d_arenah": 5}
stages = np.array([week_to_stage[w] for w in week_numbers])

combined_matrix rows: 119227
week_numbers: (119227,)


KeyboardInterrupt: 

In [ ]:
# -----------------------------
# 3. PCA + t-SNE (unchanged)
# -----------------------------
pca = PCA(n_components=50, random_state=42)
X_red = pca.fit_transform(similarity_matrix)

In [ ]:
tsne = openTSNE.TSNE(
    perplexity=30,
    initialization="pca",
    metric="cosine",
    n_jobs=8,
    random_state=3,
)

embedding = tsne.fit(X_red)  # (N, 2)

In [ ]:
# Plot with Plotly with a new tsne graph for each stage (1..5) colored by cluster_labels using the same color palette as before
labels_for_color = cluster_labels
labels_str = labels_for_color.astype(str)
unique_labels = sorted(set(labels_str))

palette = glasbey.create_palette(palette_size=len(unique_labels))
label_to_color = {lab: palette[i] for i, lab in enumerate(unique_labels)}
colors = [label_to_color[lab] for lab in labels_str]

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=embedding[:, 0],
        y=embedding[:, 1],
        mode="markers",
        marker=dict(size=5, color=colors),
        text=labels_str,  # hover text
        showlegend=False
    )
)
fig.update_layout(
    title="t-SNE of similarity matrix colored by cluster labels",
    xaxis_title="t-SNE 1",
    yaxis_title="t-SNE 2"
)
fig.show()